In [ ]:
!git clone -b ml-lr https://github.com/JKaraman93/bet.git

/home/karaman/Documents/bet


Cloning into 'bet'...
remote: Enumerating objects: 9820, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 9820 (delta 0), reused 0 (delta 0), pack-reused 9817 (from 3)
Receiving objects: 100% (9820/9820), 822.15 MiB | 24.68 MiB/s, done.
Resolving deltas: 100% (455/455), done.
Updating files: 100% (464/464), done.


In [10]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import src.utils.config as config



In [15]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
player_behavior = spark.read.parquet("./data/gold/player_behavior")
player_snapshot = spark.read.parquet("./data/gold/player_snapshot")
labels = spark.read.parquet("./data/gold/labels")


In [16]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [22]:

player_snapshot = player_snapshot.select('player_idx',
 'country',
 'age_bucket',
 #'device_type',
 #'acquisition_channel',
 #'registration_date',
 #'risk_segment',
 )

model_df = (
    player_behavior
        .join(
            player_snapshot,
            on="player_idx",
            how="left"
        )
        .join(
            labels,
            on=["player_idx", "reference_date"],
            how="inner"   # label must exist
        )
)

In [23]:
categorical_cols = [
    "country",
    "age_bucket",
    "device_type",
    "acquisition_channel",
    "risk_segment",
    "lifecycle_stage"
]
player_snapshot.select()


DataFrame[]

In [24]:
model_df.show()

+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+-------+----------+-------------+
|player_idx|reference_date|    balance_7d_ago|   balance_30d_ago|net_amount_result_7d|net_amount_result_30d|num_sessions_7d|net_game_result_7d|num_sessions_30d|avg_sessions_duration_30d|avg_bet_amount_30d|net_game_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|country|age_bucket|next_7d_churn|
+----------+--------------+------------------+------------------+--------------------+---------------------+---------------+------------------+----------------+-------------------------+------------------+-------------------+----------------------+-----------------+--------------------+----------------+-------+----------+-